# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hidatara-ds/flyrank-intern-ml-gilang/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data Contract: Grain & Time Horizon

This data contract formalizes the dataset for our **Lane 2 (Refresh / Content Opportunity Scoring)** task framed in `w02_ml_task_framing.ipynb`.

1. **Unit of Analysis (Grain):**
   - **One row = one unique pseudonymized content item (article/page)**, uniquely identified by `content_id`.
   - Each content item belongs to a pseudonymized client organization (`client_id`). There are exactly **30,000 content items** distributed across **32 clients**.
2. **Time Window:**
   - All search (Google Search Console) and web analytics (GA4) totals cover a **trailing 90-day aggregate window** ending at export time.
   - Retrospective trend tracking uses two fixed 30-day internal sub-windows: `*_last_30d` (days 1–30 back) and `*_prev_30d` (days 31–60 back).
3. **Dataset Ingestion Boundaries & Filters:**
   - `impressions_90d >= 1`: Active content with established search presence (dormant pages with 0 impressions are excluded).
   - `content_age_days >= 90`: Mature content with sufficient performance history for reliable trend analysis (no brand-new content).
   - Deduplicated on `content_id` to guarantee 1:1 row uniqueness.

In [1]:
import os, sys
import pandas as pd
import numpy as np

# Navigate to repository root if executing from work/notebooks
while not os.path.isdir("data/raw") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")

data_path = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(data_path), f"Contract input missing: {data_path}"

df = pd.read_csv(data_path)

# Verify Grain (1 row = 1 content_id)
grain_duplicates = df.groupby("content_id").size().loc[lambda x: x > 1]
assert len(grain_duplicates) == 0, f"Grain violation! Found duplicates: {len(grain_duplicates)}"

print(f"Contract Grain Check: PASSED (0 duplicate content_id rows)")
print(f"Total Content Items (Rows): {len(df):,}")
print(f"Total Pseudonymized Clients: {df['client_id'].nunique()}")
print(f"Content Age Range: {df['content_age_days'].min()} to {df['content_age_days'].max()} days (all >= 90 days)")
print(f"Minimum 90-day Impressions: {df['impressions_90d'].min()} (all >= 1 impression)")

Contract Grain Check: PASSED (0 duplicate content_id rows)
Total Content Items (Rows): 30,000
Total Pseudonymized Clients: 32
Content Age Range: 90 to 564 days (all >= 90 days)
Minimum 90-day Impressions: 1 (all >= 1 impression)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Classification Manifesto

Every column in `data/raw/content_refresh_anonymized.csv` (44 columns total) is explicitly partitioned into exactly one functional role:

| Bucket | Purpose & Boundary | Columns Included |
|---|---|---|
| **Features** | Observable signals knowable **before** the decision moment. Safe for model training. | **Numeric Search/Traffic:** `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`.<br>**Content Properties & Search Context:** `content_age_days`, `days_since_last_update`, `word_count`, `char_count`, `search_volume`, `competition`, `cpc`.<br>**Tiers & Categoricals:** `content_type`, `competition_level`, `main_intent`, `age_tier`, `age_tier_order`, `freshness_tier`, `word_count_tier`, `char_count_tier`, `impression_tier`, `position_tier`. |
| **Label / Proxy** | The target we score and predict. Derived from observed retrospective movement. | `is_declining_label` (derived as `trend_direction == 'down'`). |
| **Context** | Administrative metadata used strictly for joins, grouping, and train/test splitting. Never fed into model features. | `content_id` (unique page key), `client_id` (organization key; reserved strictly for **client-holdout validation splits**). |
| **Excluded** | Fields strictly prohibited from feature vectors due to leakage or invalid signal. | **Target Leakage (Direct Outcome):** `trend_direction`, `trend_pct` (*label is mathematically derived from these*).<br>**Target Leakage (Sub-window Components):** `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` (*sub-windows used to calculate trend_pct*).<br>**Sparse Metadata:** `provider_used`, `model_used` (*unreliable LLM generation metadata; mostly blank/unknown*). |

In [2]:
# Formal field classification contract
features_numeric = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "age_tier_order", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct"
]

features_categorical = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier"
]

context_columns = ["content_id", "client_id"]

excluded_columns = [
    # Direct Label Leakage
    "trend_direction", "trend_pct",
    # Sub-window components (Leakage)
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    # Sparse / Non-feature metadata
    "provider_used", "model_used"
]

# Verify complete partitioning of all 44 dataset columns
all_contract_columns = set(features_numeric + features_categorical + context_columns + excluded_columns)
raw_columns = set(df.columns)

assert all_contract_columns == raw_columns, f"Classification mismatch: {raw_columns.symmetric_difference(all_contract_columns)}"
print(f"Total Dataset Columns Accounted For: {len(all_contract_columns)} / {len(raw_columns)}")
print(f"Features (Safe Model Inputs):       {len(features_numeric) + len(features_categorical)} ({len(features_numeric)} numeric + {len(features_categorical)} categorical)")
print(f"Context (Keys / Split Grouping):     {len(context_columns)}")
print(f"Excluded (Leakage / Untrusted):      {len(excluded_columns)}")

Total Dataset Columns Accounted For: 44 / 44
Features (Safe Model Inputs):       32 (23 numeric + 9 categorical)
Context (Keys / Split Grouping):     2
Excluded (Leakage / Untrusted):      10


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Contract Verification Suite

To ensure data integrity and prevent subtle model failure modes, we execute four mandatory verifications:
1. **Grain Uniqueness**: Confirming zero duplicate rows on `content_id`.
2. **Missingness by Category**: Proving missingness is **not random (MNAR)**. Keyword fields (`search_volume`, `cpc`, `competition`) are 100% missing for `feedly article`, and `word_count` is 28.3% missing for `keyword article`. Blind `fillna(0)` would silently encode content types into numerical features.
3. **Domain Gotchas**:
   - `avg_position == 0` represents *no data* (1,205 pages), not position 0.
   - `scroll_rate` and `ai_traffic_pct` can exceed 100% due to cross-system numerator/denominator tracking.
   - Rate metrics (`ctr`, `engagement_rate`, etc.) are pre-multiplied percentages (x100).

In [3]:
# 1. Grain Probe
dup_count = df.duplicated(subset=["content_id"]).sum()
print(f"[Query 1: Grain Check] Duplicates on content_id: {dup_count}")

# 2. Missingness Analysis by Content Type
print("\n[Query 2: Missingness by Content Type (% missing)]")
missing_audit = (df.groupby("content_type")[["search_volume", "word_count", "cpc"]].apply(
    lambda g: g.isna().mean()
).round(4) * 100)
print(missing_audit.to_string())

# 3. Position Zero Check (1,205 rows with 'no rank data')
zero_pos_count = (df["avg_position"] == 0).sum()
print(f"\n[Query 3: Position Zero Gotcha] avg_position == 0 count: {zero_pos_count:,} ({zero_pos_count / len(df):.2%})")

# 4. Rates Exceeding 100% (Cross-system tracking gotcha)
scroll_over_100 = int((df["scroll_rate"] > 100).sum())
ai_over_100 = int((df["ai_traffic_pct"] > 100).sum())
print(f"[Query 4: Multi-system Rates] scroll_rate > 100%: {scroll_over_100} rows | ai_traffic_pct > 100%: {ai_over_100} rows")

# 5. Label Distribution Verification
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print(f"[Query 5: Target Distribution] is_declining_label base rate: {df['is_declining_label'].mean():.3%}")

[Query 1: Grain Check] Duplicates on content_id: 0

[Query 2: Missingness by Content Type (% missing)]
                    search_volume  word_count     cpc
content_type                                         
comparison article           0.00         0.0    0.00
feedly article             100.00         0.0  100.00
keyword article              1.37        28.3    1.37

[Query 3: Position Zero Gotcha] avg_position == 0 count: 1,205 (4.02%)
[Query 4: Multi-system Rates] scroll_rate > 100%: 119 rows | ai_traffic_pct > 100%: 23 rows
[Query 5: Target Distribution] is_declining_label base rate: 54.207%


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Contractual Boundaries & Honest Claims

1. **No Causal Inferences:**
   - This dataset contains observational performance signals. Predicting decline or scoring refresh opportunity is a **decision-support** ranking task.
   - We **cannot** claim that updating an article will *cause* rankings or traffic to recover; demonstrating causality requires randomized controlled experiments or difference-in-differences designs.
2. **Not Predicting Google's Proprietary Algorithm:**
   - We do not model Google's internal ranking formulas. We model empirical historical decline patterns observed across client content portfolios.
3. **Time Window Conflation (Starter Proxy Limitation):**
   - In this 90-day starter snapshot, the feature window and the 30-day trend comparison window share the same trailing timeframe. In production (the 79M-row warehouse release), features are strictly separated into prior historical windows (`T - 90d` to `T`) and evaluated on future forward windows (`T` to `T + 30d`).
4. **Severe Client Imbalance (Why Client-Holdout is Mandatory):**
   - The 30,000 pages are distributed across 32 clients with extreme imbalance: the largest client accounts for over 7,000 pages, while the smallest has only 3 pages.
   - A standard random train/test split would cause **cross-client leakage** (the model would memorize domain-specific baselines). Therefore, cross-validation must strictly use **Client-Holdout Splits (`client_id` grouped)**.

In [4]:
# Audit Client Distribution & Imbalance
client_volume = df["client_id"].value_counts()
top_5_share = client_volume.head(5).sum() / len(df)
bottom_10_share = client_volume.tail(10).sum() / len(df)

print(f"Total Clients: {len(client_volume)}")
print(f"Top Client Volume: {client_volume.iloc[0]:,} pages ({client_volume.iloc[0] / len(df):.1%})")
print(f"Smallest Client Volume: {client_volume.iloc[-1]:,} pages")
print(f"Top 5 Clients Concentration: {top_5_share:.1%} of all data")
print(f"Bottom 10 Clients Concentration: {bottom_10_share:.2%} of all data")

print("\nValidation Contract Requirement:")
print("-> Random train/test splits are PROHIBITED due to site-level domain correlation.")
print("-> Models MUST use GroupKFold / Client-Holdout splits on 'client_id'.")

Total Clients: 32
Top Client Volume: 7,008 pages (23.4%)
Smallest Client Volume: 3 pages
Top 5 Clients Concentration: 56.8% of all data
Bottom 10 Clients Concentration: 1.95% of all data

Validation Contract Requirement:
-> Random train/test splits are PROHIBITED due to site-level domain correlation.
-> Models MUST use GroupKFold / Client-Holdout splits on 'client_id'.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.